# Предобученные модели: первый семинар
Алан-Барсаг Газзаев · 14 ноября 2026

Научимся применять готовые модели через `transformers.pipeline()`: восстановление токенов, классификация текста, NER, ответы по фрагменту и суммаризация. `AutoTokenizer` поможет увидеть вход модели, а `datasets` — загрузить готовые данные. Обучение и `Trainer` относятся ко второму семинару.

Тексты заданий — на русском. Примеры посвящены кино, путешествиям, науке и городским новостям. Служба поддержки не используется как сквозная задача.


## Запуск и план на 90 минут
В Colab: **Среда выполнения → Сменить среду выполнения → T4 GPU**, затем «Выполнить все». Первый запуск скачивает веса из публичного Hugging Face Hub; API-ключ и подписка на модель не нужны. Загрузка может занять несколько минут. CPU допустим, но суммаризация будет медленнее. Оставьте несколько гигабайт свободного диска.

Локально: Python 3.10–3.12, отдельное окружение, `pip install -r requirements.txt`, `jupyter lab`. Основная проверка этого выпуска проводится в Colab; локальная совместимость отдельно не подтверждается.

План: 0–10 минут — Hub и окружение; 10–30 — токены и маски; 30–50 — классификация и метрики; 50–60 — NER; 60–75 — ответы по тексту; 75–85 — суммаризация; 85–90 — отчёт. Дополнительные изменения примеров выполняются после основного прогона.


## 1. Установка совместимых библиотек
Фиксируем Transformers 4.57.3: здесь доступен `pipeline("summarization")`. В Transformers 5 этот интерфейс изменился. Остальные версии тоже фиксируем, чтобы повторный запуск не менял поведение незаметно.

Если Colab сообщает, что уже импортированные библиотеки обновлены, выберите «Перезапустить сеанс и выполнить все». Сам ноутбук проверяет это состояние и не продолжает со смешанными версиями. PyTorch должен быть не старее 2.6 для безопасной загрузки старых весов `.bin`.


In [ ]:
import sys, subprocess
from importlib.metadata import version, PackageNotFoundError
PACKAGES = {"transformers": "4.57.3", "datasets": "4.4.1", "sentencepiece": "0.2.1", "pandas": "2.2.3", "scikit-learn": "1.6.1"}
def installed(name):
    try: return version(name)
    except PackageNotFoundError: return None
needs = [f"{name}=={wanted}" for name, wanted in PACKAGES.items() if installed(name) != wanted]
already_imported = any(name.replace("-", "_") in sys.modules for name in PACKAGES)
if needs:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *needs])
    if already_imported:
        raise RuntimeError("Библиотеки обновлены. Перезапустите сеанс и выполните все ячейки заново.")
print({name: installed(name) for name in PACKAGES})


In [ ]:
import os, json, time, gc, platform, hashlib
from pathlib import Path
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import torch
import numpy as np
import pandas as pd
from IPython.display import display
from transformers import pipeline, AutoTokenizer, AutoConfig, set_seed
from datasets import load_dataset, Dataset
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
assert tuple(map(int, torch.__version__.split("+")[0].split(".")[:2])) >= (2, 6), "Нужен PyTorch >= 2.6"
set_seed(42)
DEVICE = 0 if torch.cuda.is_available() else -1
pd.set_option("display.max_colwidth", 160)
RESULTS = {}
STARTED = time.time()
print("Python:", platform.python_version(), "PyTorch:", torch.__version__)
print("Устройство:", torch.cuda.get_device_name(0) if DEVICE == 0 else "CPU")


## 2. Модель на Hub: что именно мы загружаем
Hub хранит веса, конфигурацию, токенизатор и карточку модели. Имя репозитория не гарантирует пригодность для любой задачи. У классификатора, NER и ответов по тексту разные выходные головы.

В словаре ниже зафиксированы точные ревизии. `trust_remote_code` не требуется: используются стандартные классы Transformers. Маленькие модели выбраны для скорости, а не как лидеры качества. Для QA используем RuBERT, дообученный на русском SberQUAD. Собственные примеры остаются учебной демонстрацией, без утверждений о качестве на независимом benchmark.


In [ ]:
MODELS = {'tokens': {'id': 'cointegrated/rubert-tiny2', 'revision': 'e8ed3b0c8bbf4fb6984c3de043bf7d2f4e5969ae'}, 'sentiment': {'id': 'cointegrated/rubert-tiny-sentiment-balanced', 'revision': 'e0679dd4460e48648f5d4d819ae0f505384f5a9d'}, 'ner': {'id': 'r1char9/ner-rubert-tiny-news', 'revision': 'b5bfb41d74e1d0faa24457c7f5ebae834661b138'}, 'qa': {'id': 'MilyaShams/rubert-russian-qa-sberquad', 'revision': 'e91859a0f4e34ce2208074dd14b94d7c932614cf'}, 'summary': {'id': 'IlyaGusev/rut5_base_sum_gazeta', 'revision': 'f09a08cae5d74c70e55da1a6ebb49f88c26f433b'}}
display(pd.DataFrame([{"task": task, **spec, "card": "https://huggingface.co/" + spec["id"]} for task, spec in MODELS.items()]))
config = AutoConfig.from_pretrained(MODELS["sentiment"]["id"], revision=MODELS["sentiment"]["revision"])
print("Архитектура:", config.architectures)
print("Метки классификатора:", config.id2label)
print("Предел позиций:", config.max_position_embeddings)


## 3. Текст, токены и символьные границы
`AutoTokenizer` подбирает подходящий класс из конфигурации. `use_fast=True` позволяет получать символьные смещения. Они связывают каждый подсловный токен с исходной строкой.

Идентификатор токена — индекс в словаре, а не численная оценка смысла. Префикс `##` обозначает продолжение слова в WordPiece. У специальных токенов смещения обычно `(0, 0)`, поэтому их отдельно отмечаем в таблице.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODELS["tokens"]["id"], revision=MODELS["tokens"]["revision"], use_fast=True)
assert tokenizer.is_fast
text = "Переписывание заметок помогает запоминать материал."
encoded = tokenizer(text, return_offsets_mapping=True, return_special_tokens_mask=True)
token_table = pd.DataFrame({
    "id": encoded["input_ids"],
    "token": tokenizer.convert_ids_to_tokens(encoded["input_ids"]),
    "offset": encoded["offset_mapping"],
    "special": encoded["special_tokens_mask"],
    "fragment": [text[a:b] for a,b in encoded["offset_mapping"]],
})
display(token_table)
print("Обратное преобразование:", tokenizer.decode(encoded["input_ids"], skip_special_tokens=True))
assert len(encoded["input_ids"]) == len(encoded["offset_mapping"])
RESULTS["tokenization"] = token_table.to_dict("records")


### Упражнение: длина и нормализация
Сравним обычное слово, редкое название, опечатку и короткую фразу. Границы подслов не обязаны совпадать с морфемами. Обратное преобразование может нормализовать пробелы, поэтому не проверяем побайтовое равенство исходной строке.


In [ ]:
examples = ["университет", "электроэнцефалография", "Санкт-Петербург", "унивирситет", "Кот спит."]
display(pd.DataFrame([{"text": s, "tokens": tokenizer.tokenize(s), "n_tokens": len(tokenizer(s, add_special_tokens=False)["input_ids"])} for s in examples]))


## 4. Заполнение и маска внимания
Батч должен иметь прямоугольную форму. `padding=True` дополняет короткие входы до самого длинного. `attention_mask=0` отмечает добавленные позиции, `1` — доступные входные позиции.

Это маска заполнения для кодировщика. Она не является каузальной треугольной маской и не совпадает с заменой слова на `[MASK]`. Сами матрицы внимания вручную не реализуем.


In [ ]:
sentences = ["Кот спит.", "Студенты обсуждают новую статью о космических исследованиях."]
batch = tokenizer(sentences, padding=True, return_tensors="pt")
print("Форма input_ids:", tuple(batch["input_ids"].shape))
display(pd.DataFrame(batch["input_ids"].numpy()))
display(pd.DataFrame(batch["attention_mask"].numpy()))
assert batch["input_ids"].shape == batch["attention_mask"].shape
for i, sentence in enumerate(sentences):
    assert int(batch["attention_mask"][i].sum()) == len(tokenizer(sentence)["input_ids"])
print("Проверено: реальные токены и заполнение разделены маской.")


### Обрезка меняет данные
`max_length` измеряется в токенах вместе со специальными токенами. Ниже специально используем очень короткий предел, чтобы увидеть потерю конца текста. В заданиях дальше выбран предел, в который помещаются учебные примеры.


In [ ]:
long_text = "В начале рассказа герой отправился в путешествие. " * 8 + "В конце выяснилось, что поездка была сном."
full_ids = tokenizer(long_text)["input_ids"]
short_ids = tokenizer(long_text, truncation=True, max_length=24)["input_ids"]
print("До:", len(full_ids), "После:", len(short_ids))
print(tokenizer.decode(short_ids, skip_special_tokens=True))
assert len(short_ids) == 24 and len(full_ids) > 24
print("Последняя фраза потеряна: такой вход уже не эквивалентен исходному.")


## 5. Первый pipeline: восстановление пропуска
В `pipeline()` указываем задачу и готовую модель. Библиотека выполняет токенизацию, перенос на устройство, вызов модели и преобразование выхода.

`fill-mask` возвращает наиболее вероятные токены для одного пропуска. Это не свободная генерация предложения. Несколько вариантов могут быть грамматически и содержательно допустимыми; score не доказывает истинность утверждения.


In [ ]:
fill = pipeline("fill-mask", model=MODELS["tokens"]["id"], revision=MODELS["tokens"]["revision"], device=DEVICE)
masked = f"Студент читает {fill.tokenizer.mask_token}."
mask_answers = fill(masked, top_k=5)
display(pd.DataFrame(mask_answers))
assert len(mask_answers) == 5 and all("token_str" in x for x in mask_answers)
RESULTS["fill_mask"] = mask_answers
del fill
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


## 6. Классификация тональности
У модели три класса: negative, neutral, positive. Используем готовую голову классификации. Загружать базовый BERT как классификатор без обученной головы было бы ошибкой: библиотека могла бы создать случайные веса.

`top_k=None` показывает оценки всех классов. Здесь явно используем softmax, поскольку выбираем один класс. Это нормированные оценки модели, но их калибровка на наших текстах отдельно не проверена.


In [ ]:
sentiment = pipeline("text-classification", model=MODELS["sentiment"]["id"], revision=MODELS["sentiment"]["revision"], device=DEVICE)
reviews = ["Фильм великолепный, посмотрю его ещё раз!", "Скучный фильм, совершенно не понравился.", "Фильм длится два часа."]
all_scores = sentiment(reviews, top_k=None, function_to_apply="softmax", batch_size=8, truncation=True, max_length=256)
score_table = pd.DataFrame([{ "text": text, **{x["label"]: x["score"] for x in scores}} for text, scores in zip(reviews, all_scores)])
display(score_table)
assert len(all_scores) == len(reviews)
assert all(abs(sum(x["score"] for x in row)-1) < 1e-5 for row in all_scores)


### Небольшая проверка с заранее заданными метками
Ниже девять авторских учебных примеров, по три на класс. Эталоны фиксируем до вызова. Это демонстрационный набор, а не независимый benchmark и не основание обещать качество на всех отзывах.

Предсказания получаем одним вызовом pipeline по списку текстов. Цикл ниже только собирает таблицу — он не реализует собственный инференс.


In [ ]:
demo_data = Dataset.from_dict({
    "text": ["Прекрасная книга, прочитал с удовольствием!", "Концерт был замечательным!", "Потрясающая выставка, очень понравилась.", "Ужасная книга, зря потратил время.", "Концерт разочаровал, звук отвратительный.", "Фильм скучный и затянутый.", "Книга состоит из десяти глав.", "Концерт начнётся в семь часов.", "Выставка проходит на втором этаже."],
    "gold": ["positive"]*3 + ["negative"]*3 + ["neutral"]*3,
})
predictions = sentiment(list(demo_data["text"]), batch_size=8, truncation=True, max_length=256, function_to_apply="softmax")
classification_df = demo_data.to_pandas()
classification_df["prediction"] = [p["label"] for p in predictions]
classification_df["score"] = [p["score"] for p in predictions]
classification_df["correct"] = classification_df.gold == classification_df.prediction
display(classification_df)
assert len(predictions) == 9
RESULTS["classification"] = classification_df.to_dict("records")


### Метрики и ошибки
Accuracy — доля верных предсказаний. Macro-F1 придаёт одинаковый вес каждому классу. Матрица ошибок показывает истинные классы по строкам и предсказанные по столбцам.

Если все примеры решены, это успех на этих девяти фразах. Для следующего опыта добавьте отрицание или смешанную оценку, но не переписывайте старые эталоны под ответ модели.


In [ ]:
labels = ["negative", "neutral", "positive"]
metrics = {"n": len(classification_df), "accuracy": accuracy_score(classification_df.gold, classification_df.prediction), "macro_f1": f1_score(classification_df.gold, classification_df.prediction, labels=labels, average="macro", zero_division=0)}
print(metrics)
print(classification_report(classification_df.gold, classification_df.prediction, labels=labels, zero_division=0))
display(pd.DataFrame(confusion_matrix(classification_df.gold, classification_df.prediction, labels=labels), index=labels, columns=labels))
display(classification_df.loc[~classification_df.correct])
RESULTS["classification_metrics"] = metrics


## 7. Загрузка данных с Hugging Face Hub
`load_dataset()` загружает готовый корпус. Используем потоковый режим и берём только 30 записей для знакомства со схемой. Здесь есть тексты из разных источников, в том числе пользовательские комментарии; автоматически показываем только длины и метки.

У корпуса другая нумерация: 0 = neutral, 1 = positive, 2 = negative. Её нельзя перепутать с индексами модели. Корпус агрегирован, а пересечение с данными обучения классификатора нами не установлено. Поэтому не представляем оценку на нём как независимую проверку модели.


In [ ]:
DATASET_ID = 'MonoHime/ru_sentiment_dataset'
DATASET_REVISION = 'a66a66ee3a858c2b59e056b3fb7dfaf501fc5425'
stream = load_dataset(DATASET_ID, revision=DATASET_REVISION, split="train", streaming=True)
hub_sample = Dataset.from_list(list(stream.take(30)))
assert len(hub_sample) == 30
assert {"text", "sentiment"}.issubset(hub_sample.column_names)
print("Столбцы:", hub_sample.column_names)
label_names = {0: "neutral", 1: "positive", 2: "negative"}
hub_overview = pd.DataFrame({"characters": [len(x) for x in hub_sample["text"]], "label": [label_names[int(x)] for x in hub_sample["sentiment"]]})
display(hub_overview.head())
display(hub_overview.label.value_counts())
RESULTS["hub_dataset"] = {"id": DATASET_ID, "revision": DATASET_REVISION, "sample_size": len(hub_sample), "label_counts": hub_overview.label.value_counts().to_dict()}
del sentiment
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


## 8. NER: короткий пример разметки токенов
NER извлекает людей, организации и места. В отличие от тональности, это несколько фрагментов внутри текста. Используем `pipeline("token-classification")` и готовую агрегацию подслов `aggregation_strategy="simple"`.

Модель обучена на новостях и использует фиксированный набор меток. Она не обязана выделять любые категории, которые мы придумали. `start` и `end` задают символьные границы в исходной строке; конец не включается.


In [ ]:
ner = pipeline("token-classification", model=MODELS["ner"]["id"], revision=MODELS["ner"]["revision"], aggregation_strategy="simple", device=DEVICE)
print("Схема меток:", ner.model.config.id2label)
ner_text = "Александр Пушкин родился в Москве. Компания Яндекс открыла офис в Казани."
entities = ner(ner_text)
ner_df = pd.DataFrame(entities)
if entities:
    ner_df["fragment"] = [ner_text[e["start"]:e["end"]] for e in entities]
display(ner_df)
assert entities, "Модель не выделила ни одной сущности на демонстрационном тексте"
assert all(0 <= e["start"] < e["end"] <= len(ner_text) for e in entities)
RESULTS["ner"] = [{**e, "score": float(e["score"]), "fragment": ner_text[e["start"]:e["end"]]} for e in entities]


### Разбор результата NER
Ожидаемые смысловые фрагменты: Александр Пушкин — человек; Москва и Казань — места; Яндекс — организация. Модель может ошибиться в классе или границах. Не подменяем предсказания эталоном.

Для ручной проверки берём **исходный фрагмент по offsets**, а не только `word`: восстановленная строка токенизатора может отличаться пробелами или регистром. Измените падеж названия и обсудите, сохранились ли границы. Большой score сам по себе не доказывает правильность.


In [ ]:
del ner
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print("NER завершён; модель освобождена перед следующим заданием.")


## 9. Ответ на вопрос по фрагменту
Это извлечение ответа: модель выбирает начало и конец фрагмента контекста. Она не ищет сведения в интернете и не пишет произвольное продолжение.

Вопрос и контекст передаются в `pipeline("question-answering")`. Проверим несколько простых вопросов с заранее заданными ответами. Score не является гарантией истинности. Для этой небольшой модели не заявляем надёжного определения случаев без ответа.


In [ ]:
qa = pipeline("question-answering", model=MODELS["qa"]["id"], revision=MODELS["qa"]["revision"], device=DEVICE)
context = "Первый полёт Юрия Гагарина в космос состоялся 12 апреля 1961 года. Полёт продолжался 108 минут. Космический корабль назывался «Восток-1»."
qa_cases = [
    {"question": "Кто совершил первый полёт в космос?", "gold": "Юрия Гагарина"},
    {"question": "Сколько минут продолжался полёт?", "gold": "108 минут"},
    {"question": "Как назывался космический корабль?", "gold": "Восток-1"},
]
qa_answers = qa(question=[c["question"] for c in qa_cases], context=[context]*len(qa_cases), batch_size=3)
qa_df = pd.DataFrame([{**c, **a, "exact_match": a["answer"].strip(" «».,") == c["gold"]} for c,a in zip(qa_cases, qa_answers)])
display(qa_df)
assert len(qa_answers) == 3
assert all(context[a["start"]:a["end"]] == a["answer"] for a in qa_answers)
RESULTS["qa"] = qa_df.to_dict("records")


### Вопрос без ответа
Спросим о сведении, отсутствующем в тексте. Обычная извлекающая модель всё равно может выбрать какой-либо фрагмент. Это ожидаемое ограничение постановки, а не повод объявить выданную строку фактом.

Отдельные модели обучают на вопросах без ответа и используют специальную обработку. Одно включение флага не заменяет такое обучение и проверку порога.


In [ ]:
missing_question = "Как звали учителя Юрия Гагарина?"
missing_answer = qa(question=missing_question, context=context)
print("Вопрос:", missing_question)
print("Ответ модели:", missing_answer)
print("Эталон: в данном контексте ответа нет.")
RESULTS["qa_unanswerable"] = {"question": missing_question, "model_output": missing_answer, "gold": "нет ответа в контексте"}
del qa
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


## 10. Суммаризация готовой моделью
Теперь выход является новым текстом, а не меткой или извлечённым фрагментом. Загружаем модель, дообученную на русских новостях, через `pipeline("summarization")`.

Ниже вымышленная учебная новость. Никаких запросов к платному API нет: веса запускаются в вашей среде. Суммаризатор должен сохранить факты исходника, но может пропустить важное или добавить неподтверждённое. Проверяем содержание вручную по заранее записанным фактам.


In [ ]:
summarizer = pipeline("summarization", model=MODELS["summary"]["id"], revision=MODELS["summary"]["revision"], device=DEVICE)
article = (
    "В городе Северном открылась новая городская библиотека. Она расположена в бывшем здании школы на Лесной улице. "
    "Реконструкция здания продолжалась два года. В библиотеке собрали 25 тысяч книг, оборудовали читальный зал на 80 мест "
    "и компьютерный класс. Посетители могут брать книги домой и пользоваться электронным каталогом. "
    "Библиотека работает со вторника по воскресенье с десяти утра до восьми вечера. По понедельникам она закрыта. "
    "Вход в библиотеку бесплатный. В первый месяц сотрудники проведут встречи с писателями и занятия для школьников. "
    "Руководитель учреждения сообщила, что расписание мероприятий будет опубликовано на сайте. "
    "Городские власти рассчитывают, что новое пространство станет местом для чтения и образовательных встреч."
)
article_tokens = len(summarizer.tokenizer(article)["input_ids"])
assert article_tokens <= 512, "Учебный текст превышает выбранный предел; сократите его явно"
summary_result = summarizer(article, max_length=80, min_length=12, do_sample=False, num_beams=2, no_repeat_ngram_size=3)[0]
summary_text = summary_result["summary_text"]
print("Входных токенов:", article_tokens)
print("Резюме:", summary_text)
assert isinstance(summary_text, str) and summary_text.strip()
RESULTS["summarization"] = {"article": article, "summary": summary_text, "input_tokens": article_tokens}


### Ручная проверка резюме
Оценивайте только те утверждения, которые реально присутствуют в ответе. Суммаризатор не обязан включать все детали, но не должен их искажать.

Проверьте: открылась библиотека в Северном; 25 тысяч книг; 80 мест; бесплатный вход; понедельник — выходной. Если конкретное число пропущено, это не равнозначно неверному числу. Таблица ниже — заготовка, её пустые оценки не означают успешную проверку.


In [ ]:
summary_rubric = pd.DataFrame({"criterion": ["Сохранено главное событие", "Нет выдуманных имён, мест и дат", "Упомянутые числа совпадают с исходником", "Сохранены отрицания и ограничения", "Текст понятен без исходной новости"], "assessment": [""]*5, "evidence": [""]*5})
display(summary_rubric)
del summarizer
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


## 11. Итоги и сохранение результатов
Мы использовали готовые интерфейсы, а не писали архитектуры и циклы инференса. Однако выбор задачи, модели, токенизатора, предела длины и критериев проверки остаётся ответственностью разработчика.

Сохраним ответы, версии, ревизии моделей и метрики. Проверка исполнения подтверждает работоспособность кода. Правильность каждого ответа оценивается отдельно. Экспорт не содержит ключей, внутренних адресов или текста загруженного корпуса.


In [ ]:
OUT = Path("hf_seminar_results")
OUT.mkdir(exist_ok=True)
report = {
    "author": "Алан-Барсаг Газзаев", "models": MODELS,
    "environment": {"python": platform.python_version(), "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, "packages": {p: installed(p) for p in PACKAGES}},
    "results": RESULTS, "elapsed_seconds": round(time.time()-STARTED, 2),
    "checks": {"token_offsets": True, "padding_mask": True, "fill_mask": True, "classification": True, "hub_loading": True, "ner_spans": True, "qa_spans": True, "summary_nonempty": True},
}
(OUT/"report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2, default=lambda x: x.item() if isinstance(x, np.generic) else str(x)), encoding="utf-8")
classification_df.to_csv(OUT/"classification.csv", index=False)
ner_df.to_csv(OUT/"ner.csv", index=False)
qa_df.to_csv(OUT/"qa.csv", index=False)
summary_rubric.to_csv(OUT/"summary_rubric.csv", index=False)
import shutil
archive = shutil.make_archive("hf_seminar_results", "zip", OUT)
print("Все основные разделы выполнены.")
print("Отчёт:", str(OUT/"report.json"))
print("Архив:", archive)
print("Метрики на девяти учебных примерах:", metrics)


## Самостоятельная работа
1. Сравните токенизацию трёх редких слов и трёх опечаток. Зафиксируйте конкретные токены.
2. Добавьте к тональности четыре примера с отрицанием или смешанной оценкой. Сначала задайте эталоны, затем получите ответы.
3. В NER измените падеж имени и города. Проверьте классы и символьные границы.
4. Придумайте вопрос, ответ на который есть в контексте, и вопрос, ответа на который нет. Объясните ограничение извлечения фрагмента.
5. Измените одно число в новости и повторите суммаризацию. Проверьте, что модель не переносит число из прежнего ответа.

Во втором семинаре: полноценное разделение датасета, токенизация через Dataset.map, дообучение через Trainer и оценка на отложенной части.


## Частые проблемы
- После установки возникают ошибки импорта: перезапустите сеанс и выполните всё заново.
- Нет доступа к Hub: проверьте интернет и доступность huggingface.co; код не подменяет загрузку фиктивными результатами.
- Ошибка загрузки .bin: проверьте PyTorch >= 2.6. Не отключайте защитные проверки библиотеки.
- Нехватка GPU-памяти: перезапустите среду; ячейки освобождают модели после разделов.
- Долго загружается суммаризатор: его веса заметно больше маленьких BERT. Не запускайте несколько загрузок одной модели одновременно.
- Ответ неверный, но код работает: сохраняйте результат и разбирайте его как ошибку модели, не исправляйте молча.
- `summarization` не распознаётся: нужна закреплённая версия Transformers 4.57.3.

## Источники
[Transformers pipelines](https://huggingface.co/docs/transformers/v4.57.3/en/main_classes/pipelines) · [Datasets](https://huggingface.co/docs/datasets)

Карточки и точные ревизии всех моделей перечислены в разделе 2. [Описание корпуса тональности](https://huggingface.co/datasets/MonoHime/ru_sentiment_dataset).
